# 04 · Data Scaling & Normalization

> Part of the **Data Preprocessing** module — taught alongside the `data/customer_churn.csv` dataset.

## Learning objectives
- Know which models need scaling and which don't
- Apply Min-Max, Standard (Z-score), Robust, and MaxAbs scalers
- See the difference visually on real columns
- Avoid the #1 mistake: fitting the scaler on the test set

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

df = pd.read_csv("data/customer_churn.csv")
print("shape:", df.shape)
df.head()

shape: (5000, 15)


,customer_id,age,gender,city,education,tenure_months,contract_type,payment_method,monthly_charges,total_charges,num_products,has_credit_card,is_active_member,estimated_salary,churned
0,10000,46.0,Male,Bengaluru,Bachelors,40,Month-to-month,Electronic check,16.90,686.72,3,1,0,55083.68,1
1,10001,28.0,Female,Chennai,Masters,22,One year,Credit card,88.73,NaN,2,1,0,75325.39,0
2,10002,52.0,Female,Mumbai,Bachelors,7,Two year,Mailed check,67.45,449.61,1,1,1,57584.71,0
3,10003,54.0,Male,Delhi,Bachelors,31,One year,Bank transfer,48.61,1615.53,1,1,0,57525.63,0
4,10004,18.0,Male,Delhi,Bachelors,18,Month-to-month,Bank transfer,90.82,1612.81,1,1,0,103508.09,0


## 1. Why scale?

Many algorithms compute distances or use gradient descent. If `estimated_salary` ranges
over 0–200,000 and `age` over 18–90, the salary column dominates everything.

| Needs scaling | Doesn't care |
|---------------|--------------|
| KNN, K-means, SVM, PCA, neural nets, logistic regression with regularisation | Decision trees, random forests, gradient boosting |

Tree-based models split on thresholds — they're invariant to monotonic rescaling.

In [2]:
cols = ["age", "tenure_months", "monthly_charges", "estimated_salary"]
df[cols].describe().round(2)

,age,tenure_months,monthly_charges,estimated_salary
count,4600.00,5000.00,5000.00,4700.00
mean,41.89,35.72,65.55,73145.27
std,12.59,21.08,28.61,46857.79
min,18.00,0.00,10.00,15000.00
25%,33.00,18.00,49.97,55543.52
50%,42.00,36.00,64.32,70071.08
75%,50.00,54.00,79.22,85482.64
max,87.00,72.00,467.84,896230.97


## 2. Min-Max scaling — squish to [0, 1]

`x' = (x − min) / (max − min)`

Preserves the shape of the original distribution. Sensitive to outliers — one extreme
value pulls everything else into a narrow band.

In [6]:
from sklearn.preprocessing import MinMaxScaler

mm = MinMaxScaler()
medians = df[cols].median()
print("medians:\n", medians.round(2))
df_fixed = df[cols].fillna(medians)
transformed = mm.fit_transform(df_fixed)
print("transformed :", transformed)
X_mm = pd.DataFrame(transformed, columns=cols)
X_mm

medians:
 age                    42.00
tenure_months          36.00
monthly_charges        64.32
estimated_salary    70071.08
dtype: float64
transformed : [[0.4057971  0.55555556 0.01507077 0.04548601]
 [0.14492754 0.30555556 0.17195964 0.06845582]
 [0.49275362 0.09722222 0.12548052 0.04832412]
 ...
 [0.34782609 0.84722222 0.16119168 0.06249336]
 [0.36231884 0.55555556 0.06120042 0.10547827]
 [0.17391304 0.31944444 0.14793378 0.04379202]]


,age,tenure_months,monthly_charges,estimated_salary
0,0.405797,0.555556,0.015071,0.045486
1,0.144928,0.305556,0.171960,0.068456
2,0.492754,0.097222,0.125481,0.048324
3,0.521739,0.430556,0.084331,0.048257
4,0.000000,0.250000,0.176525,0.100437
...,...,...,...,...
4995,0.550725,0.736111,0.078434,0.054983
4996,0.289855,0.958333,0.111589,0.079416
4997,0.347826,0.847222,0.161192,0.062493
4998,0.362319,0.555556,0.061200,0.105478


## 3. Standardisation (Z-score)

`x' = (x − mean) / std`

Centres at 0, unit variance. Doesn't bound the output. The default choice for most
linear models, SVMs, and neural nets.

In [7]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
medians = df[cols].median()
print("medians:\n", medians.round(2))
df_fixed = df[cols].fillna(medians)
transformed = ss.fit_transform(df_fixed)
print("transformed :", transformed) 
X_ss = pd.DataFrame(transformed, columns=cols)
X_ss.describe().round(3)

medians:
 age                    42.00
tenure_months          36.00
monthly_charges        64.32
estimated_salary    70071.08
dtype: float64
transformed : [[ 0.33992578  0.20314572 -1.70062276 -0.39349794]
 [-1.1509041  -0.65080641  0.81051521  0.05204711]
 [ 0.83686907 -1.36243319  0.06657791 -0.33844718]
 ...
 [ 0.00863025  1.19942322  0.63816507 -0.06360664]
 [ 0.09145413  0.20314572 -0.96227898  0.77017094]
 [-0.98525633 -0.60336463  0.42596116 -0.42635632]]


,age,tenure_months,monthly_charges,estimated_salary
count,5000.000,5000.000,5000.000,5000.000
mean,-0.000,-0.000,-0.000,-0.000
std,1.000,1.000,1.000,1.000
min,-1.979,-1.695,-1.942,-1.276
25%,-0.654,-0.841,-0.545,-0.355
50%,0.009,0.013,-0.043,-0.064
75%,0.588,0.867,0.478,0.253
max,3.736,1.721,14.064,18.121


## 4. Robust scaling

`x' = (x − median) / IQR`

Uses median + IQR instead of mean + std → unaffected by outliers. Great when you have
extreme values you don't want to remove.

In [8]:
from sklearn.preprocessing import RobustScaler

rs = RobustScaler()
medians = df[cols].median()
print("medians:\n", medians.round(2))
df_fixed = df[cols].fillna(medians)
transformed = rs.fit_transform(df_fixed)
print("transformed :", transformed)
X_rs = pd.DataFrame(transformed, columns=cols)
X_rs.describe().round(3)

medians:
 age                    42.00
tenure_months          36.00
monthly_charges        64.32
estimated_salary    70071.08
dtype: float64
transformed : [[ 0.26666667  0.11111111 -1.62122895 -0.54268584]
 [-0.93333333 -0.38888889  0.83428767  0.19025555]
 [ 0.66666667 -0.80555556  0.10682848 -0.45212489]
 ...
 [ 0.          0.69444444  0.66575506  0.        ]
 [ 0.06666667  0.11111111 -0.89923938  1.37160112]
 [-0.8        -0.36111111  0.45825143 -0.59673932]]


,age,tenure_months,monthly_charges,estimated_salary
count,5000.000,5000.000,5000.000,5000.000
mean,-0.007,-0.008,0.042,0.105
std,0.805,0.586,0.978,1.645
min,-1.600,-1.000,-1.857,-1.994
25%,-0.533,-0.500,-0.491,-0.480
50%,0.000,0.000,0.000,0.000
75%,0.467,0.500,0.509,0.520
max,3.000,1.000,13.794,29.915


## 5. Visual comparison

Look at `estimated_salary` (which has heavy outliers) under each scaler.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
series = [
    ("Original",       df["estimated_salary"].fillna(df["estimated_salary"].median())),
    ("Min-Max",        X_mm["estimated_salary"]),
    ("Standardised",   X_ss["estimated_salary"]),
    ("Robust",         X_rs["estimated_salary"]),
]
for ax, (title, s) in zip(axes, series):
    ax.hist(s, bins=40); ax.set_title(title)
plt.tight_layout(); plt.show()

Notice how the Min-Max plot squashes everything to the left because of the outliers,
while Robust keeps the bulk of the data spread out.

## 6. MaxAbs and Normalizer

- **`MaxAbsScaler`** — divides by the maximum absolute value. Maps to [-1, 1] without
  shifting. Good for sparse data (it doesn't break sparsity).
- **`Normalizer`** — scales each *row* (sample) to unit length. Different beast: useful
  for text vectors / cosine similarity, almost never for tabular data.

In [ ]:
from sklearn.preprocessing import MaxAbsScaler, Normalizer

MaxAbsScaler().fit_transform(df[cols].fillna(0))[:3]

## 7. The #1 mistake — leaking through the scaler

```python
# WRONG — fits on whole dataset, leaks test stats into training
X_scaled = StandardScaler().fit_transform(X)
X_train, X_test = train_test_split(X_scaled, ...)

# RIGHT — fit on train only, transform both
X_train, X_test = train_test_split(X, ...)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)
```

The cleanest version is to put the scaler inside a `Pipeline` so it's fit per
cross-validation fold automatically.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf",   LogisticRegression(max_iter=1000)),
])

X_num = df[cols].fillna(df[cols].median())
scores = cross_val_score(pipe, X_num, df["churned"], cv=5, scoring="roc_auc")
print(f"ROC-AUC: {scores.mean():.3f}  ± {scores.std():.3f}")

## 8. Cheat sheet

| Scenario | Use |
|----------|-----|
| Default for most ML | `StandardScaler` |
| Bounded output needed (neural nets, image-like) | `MinMaxScaler` |
| Outliers present, can't remove | `RobustScaler` |
| Sparse data (text, recommender) | `MaxAbsScaler` |
| Tree-based model | none — skip scaling |

## Exercise
1. Train KNN on the dataset *without* scaling, then *with* `StandardScaler`.
   Compare ROC-AUC. Why is the difference so large?
2. Repeat with a `RandomForestClassifier`. Explain why the difference is tiny.